# Feature Construction: Engineering Domain-Informed Variables

## 1. Clear Overview

Feature construction is the creative process of engineering new variables from existing data to better represent the underlying phenomena being modeled. Unlike feature extraction (which algorithmically distills or compresses data using tools like PCA), feature construction is a domain-driven task. 

It involves inventing features that make relationships explicit, effectively translating raw data into a language that machine learning models can understand to reveal hidden signals.

In [ ]:
# Always start with imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Scikit-Learn for modeling and evaluation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder

# Set display options for pandas
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seed for reproducibility
np.random.seed(42)

print("Step 1: All necessary libraries successfully imported!")

## 2. Structured Table of Contents

- **Synthetic Data Creation**: Building our Real Estate Dataset
- **Why Feature Construction Matters**: Setting the Baseline
- **Core Strategy 1**: Mathematical Combinations
- **Core Strategy 2**: Aggregations and Context
- **Core Strategy 3**: Interaction Features
- **Core Strategy 4**: Domain-Specific Logic
- **Evaluating the Impact**: Proving the Features Work
- **Preventing Data Leakage**: The Engineering Standard
- **Visualization Gallery**: Exploring Constructed Features
- **Practice Exercises**: Apply your knowledge
- **Application Summary**: Best Practices

## 3. Synthetic Data Creation (Real Estate)

To demonstrate the power of feature construction, we need raw data. We will simulate a Real Estate dataset containing raw attributes of houses. 

Our features will include:
- `area_sqft`: Inside living area
- `lot_size_sqft`: Total land area
- `year_built`: Year construction was finished
- `quality_score`: Inspector rating from 1 to 10
- `neighborhood`: City district
- `price`: The target variable

In [ ]:
# Step 2: Create synthetic real estate dataset
n_samples = 2000

# Base features
area_sqft = np.random.normal(2000, 600, n_samples).clip(800, 6000)
lot_size_sqft = area_sqft * np.random.uniform(1.2, 5.0, n_samples) # Lot is always bigger than house
year_built = np.random.randint(1950, 2024, n_samples)
quality_score = np.random.randint(1, 11, n_samples)
neighborhoods = np.random.choice(['Downtown', 'Suburb_North', 'Suburb_South', 'Rural_East'], n_samples)

# Formulate the true price using a complex, non-linear hidden relationship
base_price = 50000
price = base_price + (area_sqft * 150) + (quality_score * 25000)

# Age depreciation factor (older houses lose value, unless historic)
age = 2024 - year_built
price -= (age * 1200)
price = np.where(year_built < 1960, price + 40000, price) # Historic premium

# Neighborhood multipliers
multipliers = {'Downtown': 1.5, 'Suburb_North': 1.2, 'Suburb_South': 0.9, 'Rural_East': 0.7}
price *= np.array([multipliers[n] for n in neighborhoods])

# Add irreducible market noise
price += np.random.normal(0, 30000, n_samples)

# Assemble DataFrame
df_raw = pd.DataFrame({
    'area_sqft': area_sqft.round(),
    'lot_size_sqft': lot_size_sqft.round(),
    'year_built': year_built,
    'quality_score': quality_score,
    'neighborhood': neighborhoods,
    'price': price.round()
})

print("Synthetic Real Estate Dataset Created!\n")
print(df_raw.head())
print(f"\nDataset Shape: {df_raw.shape}")

## 4. Why Feature Construction Matters: Setting a Baseline

Machine learning models are only as good as the features they ingest. Raw inputs often lack the explicit context necessary to solve complex problems. 

To prove this, let's train a standard Linear Regression model on the strictly RAW data and measure its performance. We will use this baseline to measure the impact of our engineered features later.

In [ ]:
# Prepare raw data for modeling (One-Hot Encoding categorical features)
df_model_raw = pd.get_dummies(df_raw, columns=['neighborhood'], drop_first=True)

X_raw = df_model_raw.drop('price', axis=1)
y_raw = df_model_raw['price']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# Train Baseline Model
baseline_model = Ridge(alpha=1.0)
baseline_model.fit(X_train_raw, y_train)

# Evaluate
raw_preds = baseline_model.predict(X_test_raw)
baseline_r2 = r2_score(y_test, raw_preds)
baseline_mae = mean_absolute_error(y_test, raw_preds)

print("--- BASELINE MODEL PERFORMANCE (RAW FEATURES) ---")
print(f"R-Squared (Accuracy): {baseline_r2:.4f}")
print(f"Mean Absolute Error:  ${baseline_mae:,.2f}")
print("\nKeep these numbers in mind. We will beat them using Feature Construction.")

## 5. Core Strategy 1: Mathematical Combinations

Applying arithmetic operations (ratios, differences, products) reveals explicit relationships that linear models cannot easily figure out on their own.

**Constructed Features:**
1. `house_age`: A linear model doesn't know that '2024 - year_built' represents age. We must construct it.
2. `yard_size`: The difference between the lot size and the house footprint.

In [ ]:
# Create a copy of our raw data to begin adding features
df_engineered = df_raw.copy()

current_year = 2024

# Feature 1: House Age
df_engineered['house_age'] = current_year - df_engineered['year_built']

# Feature 2: Yard Size (Lot Size minus Area)
df_engineered['yard_size_sqft'] = df_engineered['lot_size_sqft'] - df_engineered['area_sqft']

print("Mathematical Combination Features Added!")
print(df_engineered[['year_built', 'house_age', 'lot_size_sqft', 'area_sqft', 'yard_size_sqft']].head())

## 6. Core Strategy 2: Interaction Features

Interaction features model dependencies between two separate variables by multiplying them together to capture synergistic effects.

**Constructed Feature:**
1. `quality_area_interaction`: A high-quality score is much more valuable on a massive 5000 sqft house than it is on a tiny 800 sqft apartment. Multiplying them creates a holistic 'Total Quality Volume' metric.

In [ ]:
# Feature 3: Quality-Area Interaction
df_engineered['quality_area_interaction'] = df_engineered['quality_score'] * df_engineered['area_sqft']

# Let's visualize why this works by looking at the correlation to target
corr_quality = df_engineered['quality_score'].corr(df_engineered['price'])
corr_interaction = df_engineered['quality_area_interaction'].corr(df_engineered['price'])

print(f"Correlation of Raw Quality to Price:       {corr_quality:.3f}")
print(f"Correlation of Interaction Feature to Price: {corr_interaction:.3f}")

print("\nInsight: The interaction feature is significantly more correlated with the target variable!")

## 7. Core Strategy 3: Aggregations and Context

Computing summary statistics for groups provides vital context. An 1800 sqft house is 'large' in the city center, but 'small' in rural areas.

**Constructed Feature:**
1. `relative_size_ratio`: The ratio of the house's area compared to the average area in its specific neighborhood.

In [ ]:
# Calculate the average area per neighborhood
neighborhood_avg_area = df_engineered.groupby('neighborhood')['area_sqft'].transform('mean')

# Feature 4: Relative Size Ratio
df_engineered['relative_size_ratio'] = df_engineered['area_sqft'] / neighborhood_avg_area

print("Contextual Aggregation Features Added!")
print("Notice how homes in different neighborhoods get judged against their local peers.")
print(df_engineered[['neighborhood', 'area_sqft', 'relative_size_ratio']].head())

## 8. Core Strategy 4: Domain-Specific Logic & Categorization

Leveraging specialized knowledge allows you to build standard industry metrics or handle nonlinear behaviors. We know that very old houses often appreciate in value due to 'historic charm', breaking the linear rule of depreciation.

**Constructed Feature:**
1. `is_historic`: A binary flag representing if a house is over 60 years old.

In [ ]:
# Feature 5: Domain Flag for Historic Homes
df_engineered['is_historic'] = np.where(df_engineered['house_age'] > 60, 1, 0)

print("Historic Flag counts:")
print(df_engineered['is_historic'].value_counts())

print("\nInsight: We have explicitly taught the model to identify older homes as a distinct category.")

## 9. Preventing Data Leakage in Feature Engineering

> **Warning:** Calculating grouped statistics (like neighborhood averages) on the ENTIRE dataset before Train/Test splitting causes data leakage. Information from the test set bleeds into the training set.

The Engineering Standard: We must split the data first, then engineer features based ONLY on training statistics.

In [ ]:
# We split our original raw dataset again to do this perfectly clean
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    df_raw.drop('price', axis=1), df_raw['price'], test_size=0.2, random_state=42
)

def engineer_features(df_input, train_stats=None):
    df = df_input.copy()
    
    # 1. Math
    df['house_age'] = 2024 - df['year_built']
    df['yard_size_sqft'] = df['lot_size_sqft'] - df['area_sqft']
    
    # 2. Interactions
    df['quality_area_interaction'] = df['quality_score'] * df['area_sqft']
    
    # 3. Domain
    df['is_historic'] = np.where(df['house_age'] > 60, 1, 0)
    
    # 4. Aggregations (Crucial Leakage Prevention Step)
    if train_stats is None:
        # If this is the training set, compute the stats and save them
        train_stats = df.groupby('neighborhood')['area_sqft'].mean().to_dict()
    
    # Map the trained statistics to whatever data is passed in (train or test)
    df['local_avg_area'] = df['neighborhood'].map(train_stats)
    df['relative_size_ratio'] = df['area_sqft'] / df['local_avg_area']
    df = df.drop('local_avg_area', axis=1)
    
    return df, train_stats

# Apply to train safely
X_train_eng, saved_stats = engineer_features(X_train_clean)

# Apply to test safely using the SAVED train stats
X_test_eng, _ = engineer_features(X_test_clean, train_stats=saved_stats)

print("Safely engineered features without data leakage!")

## 10. Evaluating the Impact: Proving the Features Work

Let's train a new model on our engineered dataset and compare it to the baseline.

In [ ]:
# One-Hot encode the neighborhood feature for both train and test
X_train_final = pd.get_dummies(X_train_eng, columns=['neighborhood'], drop_first=True)
X_test_final = pd.get_dummies(X_test_eng, columns=['neighborhood'], drop_first=True)

# Ensure alignment (in case a neighborhood was missing in test)
X_train_final, X_test_final = X_train_final.align(X_test_final, join='left', axis=1, fill_value=0)

# Train Advanced Model
advanced_model = Ridge(alpha=1.0)
advanced_model.fit(X_train_final, y_train_clean)

# Evaluate
adv_preds = advanced_model.predict(X_test_final)
adv_r2 = r2_score(y_test_clean, adv_preds)
adv_mae = mean_absolute_error(y_test_clean, adv_preds)

print("--- MODEL PERFORMANCE COMPARISON ---")
print(f"Baseline R-Squared:    {baseline_r2:.4f}")
print(f"Engineered R-Squared:  {adv_r2:.4f}")
print(f"Improvement:           +{(adv_r2 - baseline_r2):.4f}")
print("------------------------------------")
print(f"Baseline MAE:          ${baseline_mae:,.2f}")
print(f"Engineered MAE:        ${adv_mae:,.2f}")
print(f"Error Reduction:       ${(baseline_mae - adv_mae):,.2f}")

print("\nMassive Success! By translating domain knowledge into math, we drastically reduced our prediction errors.")

## 11. Visualization Gallery

Let's visualize exactly why these engineered features are so powerful by charting them against the target variable (Price).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top-Left: Raw vs Constructed Age
sns.scatterplot(data=df_engineered, x='year_built', y='price', alpha=0.3, ax=axes[0, 0], color='gray')
axes[0, 0].set_title('Raw Year Built vs Price (Confusing)', fontsize=14)

# Top-Right: The Historic Flag in Action
sns.boxplot(data=df_engineered, x='is_historic', y='price', ax=axes[0, 1], palette='Set2')
axes[0, 1].set_title('Domain Feature: Historic vs Non-Historic Value', fontsize=14)

# Bottom-Left: Interaction Feature
sns.scatterplot(data=df_engineered, x='quality_area_interaction', y='price', alpha=0.5, ax=axes[1, 0], color='purple')
axes[1, 0].set_title('Interaction: Quality * Area (Highly Linear)', fontsize=14)

# Bottom-Right: Context Feature
sns.scatterplot(data=df_engineered, x='relative_size_ratio', y='price', hue='neighborhood', alpha=0.6, ax=axes[1, 1], palette='tab10')
axes[1, 1].set_title('Context: Relative Size Ratio vs Price', fontsize=14)

plt.tight_layout()
plt.show()

## 12. Practice Exercises

Now it is your turn to construct features. 

### Exercise 1: Ratio Construction

**Task:** 
Create a new feature in the `df_engineered` dataframe called `lot_to_area_ratio`. This represents how much total land the house sits on compared to the size of the house itself. 
Print the top 5 rows containing only `area_sqft`, `lot_size_sqft`, and your new feature.

In [ ]:
# --- EXERCISE 1 SOLUTION ---

df_engineered['lot_to_area_ratio'] = df_engineered['lot_size_sqft'] / df_engineered['area_sqft']

print("Lot to Area Ratio Computed:")
print(df_engineered[['area_sqft', 'lot_size_sqft', 'lot_to_area_ratio']].head())

### Exercise 2: Validating Your Construction

**Task:**
An engineered feature is only successful if it relates to the target. 
Calculate and print the Pearson correlation between your new `lot_to_area_ratio` and `price`.

In [ ]:
# --- EXERCISE 2 SOLUTION ---

correlation = df_engineered['lot_to_area_ratio'].corr(df_engineered['price'])

print(f"Correlation between Lot-to-Area Ratio and Price: {correlation:.4f}")

if abs(correlation) < 0.1:
    print("\nAnalysis: The correlation is weak. In a real project, this feature might not be useful to the model.")
    print("Not all constructed features are winners. This is why testing is crucial!")

## 13. Application Summary and Best Practices

Feature construction is an iterative, art-meets-science endeavor.

- **Leverage Domain Expertise:** The best features come from talking to subject matter experts (SMEs). A doctor knows which blood pressure ratio matters; a realtor knows historic thresholds.
- **Support Simpler Models:** Explicitly calculating variables allows interpretable models (like Linear Regression) to achieve the performance of complex black-box networks.
- **Guard Against Overfitting:** Adding 100 random interactions introduces noise. Only keep features that improve cross-validated validation scores.
- **Prevent Leakage:** Never aggregate features on the entire dataset. Always split into train/test, derive statistical rules from the train, and apply them to the test.
- **ROI:** By prioritizing meaningful construction, you often achieve greater performance gains than by simply tuning hyperparameter settings.

In [ ]:
print("-----------------------------------------------------------")
print("Notebook Execution Complete.")
print("You have successfully mastered Feature Construction techniques!")
print("-----------------------------------------------------------")